# position-correction-harness — verification

# Imports

In [ ]:
import json
import os
import sys

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

from uavloc.builds import build_dir
from uavloc.data.drone_frames import load_frames
from uavloc.data.satellite import GeoImage
from uavloc.features.segmentation import (CLASSES, SegmentationFeatures, class_probabilities, load_unet,
                                          recalibrate_batchnorm)
from uavloc.data.seg_crops import drone_crop, jittered_input, satellite_crop
from uavloc.features.vgg11 import Vgg11Features, load_vgg11
from uavloc.geo import EARTH_RADIUS_M, LatLon, haversine_m
from uavloc.matching.benchmark import injected_priors
from uavloc.matching.correlator import CorrelatorConfig, PositionCorrelator
from uavloc.track import REPO_ROOT, set_seed

# Params

In [ ]:
SLUG = "position-correction-harness"
DEVICE = "mps"
SEED = 42

SAT_IMAGE = "../archive/uav_localization_2025/data/satelites/satellite_gonchary.jpg"
SAT_TOP_LEFT = (49.762965, 24.166472)
SAT_BOTTOM_RIGHT = (49.759291, 24.174046)
ARCHIVE_2025 = "../archive/uav_localization_2025"
FRAMES_DATASET = "drone-2025-easy"
VILLAGE_DATASET = "ua-village-annotated"
DRONE_DATASET = "ua-drone-annotated"

VGG11_CHECKPOINT = "../archive/uav_localization_2025/feature_models/models_finetrain/vgg11_drop_margin08_adamw_model_20250521_034900.pth"
DRONE_INPUT_PX = 448
SEG_CHECKPOINT = "pocs/village-seg-baseline/model-round2.pt"
SEG_BN_SATELLITE = "pocs/village-seg-baseline/model-round2-bn-satellite.pt"
SEG_BN_DRONE = "pocs/village-seg-baseline/model-round2-bn-drone.pt"
SEG_SPLITS = "pocs/village-seg-baseline/results-round2.json"
DRONE_SEG_GSD_M = 0.10
MATCH_GSD_M = 0.385

SEARCH_SCALE = 1.7
FOV_W_DEG = 60.0
SAT_FEATURE_CROP_PCT = 0.03
NOISE_M = 15.0
N_DIRECTIONS = 8
REGRESSION_FRAME_STEP = 9
REGRESSION_DIRECTIONS = 4

BN_BATCHES = 60
SAT_BN_CROP_PX, SAT_BN_BATCH = 512, 4
DRONE_BN_CROP_PX, DRONE_BN_BATCH = 256, 8
RARE_CLASS_P = 0.3
BIRKY_VAL_GAP_PX = 64
IGNORE_LABEL = 255
ROUND2_MIOU = {"drone-val": 0.786, "bushcha": 0.573}
POSE_FIGURE_FRAMES = 6

# Data

In [ ]:
set_seed(SEED)
FIGURES = build_dir(SLUG, create=True) / "figures"
DERIVED = REPO_ROOT / "data" / "derived"
ARCHIVE = REPO_ROOT / ARCHIVE_2025

In [ ]:
frames = load_frames(FRAMES_DATASET)
satellite = GeoImage.load(REPO_ROOT / SAT_IMAGE, LatLon(*SAT_TOP_LEFT), LatLon(*SAT_BOTTOM_RIGHT))
len(frames), satellite.pixels.shape, satellite.mpp_xy

In [ ]:
splits = json.loads((REPO_ROOT / SEG_SPLITS).read_text())["splits"]
village_records = [json.loads(line) for line in (DERIVED / VILLAGE_DATASET / "samples.jsonl").read_text().splitlines() if line.strip()]
drone_records = {r["location"]: r for r in (json.loads(line) for line in (DERIVED / DRONE_DATASET / "samples.jsonl").read_text().splitlines() if line.strip())}
len(village_records), len(drone_records)

# Check: geo

In [ ]:
round_trip_m = [haversine_m(f.position, satellite.to_latlon(*satellite.to_px(f.position))) for f in frames]
max(round_trip_m)

In [ ]:
def haversine_2025_m(a: LatLon, b: LatLon) -> float:
    earth_radius_2025_m = 6_371_000
    return haversine_m(a, b) * earth_radius_2025_m / EARTH_RADIUS_M

In [ ]:
priors = injected_priors(frames[0].position, NOISE_M, N_DIRECTIONS, phase_deg=0.0)
injected_m = [haversine_m(frames[0].position, p) for _, p in priors]
min(injected_m), max(injected_m), haversine_2025_m(frames[0].position, priors[0][1])

# Check: regression against the 2025 PositionCorrelator

In [ ]:
sys.path.insert(0, str(ARCHIVE))
from image_matching.position_correlator import PositionCorrelator as PositionCorrelator2025  # noqa: E402

In [ ]:
class IsotropicGeoImage(GeoImage):
    """The 2025 assumption: one mean m/px for both axes."""

    @property
    def mpp_xy(self) -> tuple[float, float]:
        mpp_x, mpp_y = super().mpp_xy
        return ((mpp_x + mpp_y) / 2,) * 2

In [ ]:
vgg11 = load_vgg11(REPO_ROOT / VGG11_CHECKPOINT, "cpu")
satellite_2025 = IsotropicGeoImage(satellite.pixels, satellite.top_left, satellite.bottom_right)
heatmaps_2025 = {}
correlator_2025 = PositionCorrelator2025(str(REPO_ROOT / SAT_IMAGE), SAT_TOP_LEFT, SAT_BOTTOM_RIGHT, vgg11, DRONE_INPUT_PX,
                                         SEARCH_SCALE, "cpu", heatmap_debug_hook=lambda h, *_: heatmaps_2025.setdefault("h", h))
correlator_port = PositionCorrelator(Vgg11Features(vgg11, satellite_2025, DRONE_INPUT_PX, SEARCH_SCALE, "cpu"),
                                     CorrelatorConfig(SEARCH_SCALE, FOV_W_DEG))

In [ ]:
regression = []
for frame in frames[::REGRESSION_FRAME_STEP]:
    for bearing_deg, prior in injected_priors(frame.position, NOISE_M, REGRESSION_DIRECTIONS, phase_deg=20.0):
        heatmaps_2025.clear()
        (lat, lon), _, _ = correlator_2025.get_correlated_coordinates(frame.image(), prior, frame.yaw_deg, frame.altitude_m)
        port = correlator_port.correct(frame, prior)
        regression.append({"frame": frame.name, "bearing_deg": bearing_deg,
                           "position_diff_m": haversine_m(LatLon(lat, lon), port.position),
                           "heatmap_r": np.corrcoef(heatmaps_2025["h"].ravel(), port.heatmap.ravel())[0, 1],
                           "peak_z": port.peak_z})
regression = pd.DataFrame(regression)
regression.describe().loc[["50%", "min", "max"]]

In [ ]:
regression.sort_values("position_diff_m", ascending=False).head(5)

# BN checkpoints

In [ ]:
def satellite_training_images() -> list[np.ndarray]:
    train = {tuple(pair) for pair in splits["sat_train"]}
    birky_val_end = splits["birky_val_rows"][1] + BIRKY_VAL_GAP_PX
    images = []
    for r in village_records:
        if (r["location"], r["date"]) in train:
            image = np.asarray(Image.open(DERIVED / r["image_file"]).convert("RGB"))
            images.append(image[birky_val_end:] if r["location"].startswith("birky") else image)
    return images

In [ ]:
def drone_training_samples() -> list[tuple[np.ndarray, np.ndarray, float]]:
    samples = []
    for name in splits["drone_train"]:
        r = drone_records[name]
        samples.append((np.asarray(Image.open(DERIVED / r["image_file"]).convert("RGB")),
                        np.asarray(Image.open(DERIVED / DRONE_DATASET / r["label_file"])), r["mpp"]))
    return samples

In [ ]:
rng = np.random.default_rng(SEED)
sat_images, drone_samples = satellite_training_images(), drone_training_samples()
sat_batches = [torch.from_numpy(np.stack([jittered_input(satellite_crop(sat_images[rng.integers(len(sat_images))], SAT_BN_CROP_PX, rng), rng)
                                          for _ in range(SAT_BN_BATCH)])) for _ in range(BN_BATCHES)]
drone_batches = [torch.from_numpy(np.stack([jittered_input(drone_crop(*drone_samples[rng.integers(len(drone_samples))], rng,
                                                                      DRONE_SEG_GSD_M, DRONE_BN_CROP_PX, RARE_CLASS_P), rng)
                                            for _ in range(DRONE_BN_BATCH)])) for _ in range(BN_BATCHES)]
len(sat_images), len(drone_samples)

In [ ]:
round2 = load_unet(REPO_ROOT / SEG_CHECKPOINT, DEVICE)
seg_satellite = recalibrate_batchnorm(round2, sat_batches, DEVICE)
seg_drone = recalibrate_batchnorm(round2, drone_batches, DEVICE)
torch.save(seg_satellite.state_dict(), REPO_ROOT / SEG_BN_SATELLITE)
torch.save(seg_drone.state_dict(), REPO_ROOT / SEG_BN_DRONE)

# Check: segmentation reproduces round 2

In [ ]:
def uncovered(shape: tuple[int, int], offset_px: tuple[int, int]) -> np.ndarray:
    """Strip a date's alignment shift left without imagery, as round 2 masked it."""
    dx, dy = offset_px
    h, w = shape
    mask = np.zeros(shape, bool)
    if dx > 0: mask[:, w - dx:] = True
    if dx < 0: mask[:, :-dx] = True
    if dy > 0: mask[h - dy:, :] = True
    if dy < 0: mask[:-dy, :] = True
    return mask

In [ ]:
def miou(predictions: list[np.ndarray], labels: list[np.ndarray]) -> float:
    """Mean over classes present in the labels; intersections and unions summed over the set."""
    ious = []
    for c in range(len(CLASSES)):
        inter = sum(((p == c) & (t == c)).sum() for p, t in zip(predictions, labels))
        union = sum((((p == c) | (t == c)) & (t != IGNORE_LABEL)).sum() for p, t in zip(predictions, labels))
        if sum((t == c).sum() for t in labels):
            ious.append(inter / union)
    return float(np.mean(ious))

In [ ]:
def drone_val_pair(name: str) -> tuple[np.ndarray, np.ndarray]:
    r = drone_records[name]
    image, label = Image.open(DERIVED / r["image_file"]).convert("RGB"), Image.open(DERIVED / DRONE_DATASET / r["label_file"])
    size = (round(image.width * r["mpp"] / DRONE_SEG_GSD_M), round(image.height * r["mpp"] / DRONE_SEG_GSD_M))
    return np.asarray(image.resize(size, Image.LANCZOS)), np.asarray(label.resize(size, Image.NEAREST))

In [ ]:
def bushcha_pairs() -> list[tuple[np.ndarray, np.ndarray]]:
    pairs = []
    for r in village_records:
        if r["location"] == splits["sat_holdout_village"]:
            label = np.array(Image.open(DERIVED / VILLAGE_DATASET / r["label_file"]))
            label[uncovered(label.shape, r["image_offset_px"])] = IGNORE_LABEL
            pairs.append((np.asarray(Image.open(DERIVED / r["image_file"]).convert("RGB")), label))
    return pairs

In [ ]:
eval_sets = {"drone-val": (seg_drone, [drone_val_pair(n) for n in splits["drone_val"]]),
             "bushcha": (seg_satellite, bushcha_pairs())}
seg_scores = {}
for name, (model, pairs) in eval_sets.items():
    predictions = [class_probabilities(model, image, DEVICE).argmax(0) for image, _ in pairs]
    seg_scores[name] = {"miou": miou(predictions, [label for _, label in pairs]), "round2": ROUND2_MIOU[name]}
pd.DataFrame(seg_scores).T

# Check: identity

In [ ]:
class SatelliteAsDrone:
    """Feature extractor whose drone map is cut from the satellite at the true pose: correction must be exact."""

    def __init__(self, inner, truth: LatLon, yaw_deg: float, drone_from_satellite) -> None:
        self.inner, self.truth, self.yaw_deg, self.drone_from_satellite = inner, truth, yaw_deg, drone_from_satellite

    def drone(self, image: Image.Image, width_m: float, height_m: float) -> torch.Tensor:
        return self.drone_from_satellite(self.inner, self.truth, self.yaw_deg, width_m, height_m)

    def satellite_patch(self, center: LatLon, yaw_deg: float, width_m: float, height_m: float) -> torch.Tensor:
        return self.inner.satellite_patch(center, yaw_deg, width_m, height_m)

In [ ]:
def identity_errors(inner, drone_from_satellite, config: CorrelatorConfig) -> pd.DataFrame:
    rows = []
    for frame in frames[::REGRESSION_FRAME_STEP]:
        correlator = PositionCorrelator(SatelliteAsDrone(inner, frame.position, frame.yaw_deg, drone_from_satellite), config)
        for bearing_deg, prior in injected_priors(frame.position, NOISE_M, N_DIRECTIONS, phase_deg=0.0):
            rows.append({"frame": frame.name, "bearing_deg": bearing_deg, "reachable": correlator.reachable(frame, prior),
                         "error_m": haversine_m(correlator.correct(frame, prior).position, frame.position)})
    return pd.DataFrame(rows)

In [ ]:
vgg11_features = Vgg11Features(vgg11.to(DEVICE), satellite, DRONE_INPUT_PX, SEARCH_SCALE, DEVICE)
seg_features = SegmentationFeatures(seg_drone, seg_satellite, satellite, "all", DRONE_SEG_GSD_M, MATCH_GSD_M, DEVICE)
config = CorrelatorConfig(SEARCH_SCALE, FOV_W_DEG, sat_feature_crop_pct=SAT_FEATURE_CROP_PCT)

In [ ]:
identity = {
    "vgg11": identity_errors(vgg11_features, lambda f, c, y, w, h: f.drone(Image.fromarray(satellite.crop_rotated(c, y, w, h)), w, h), config),
    "segmentation": identity_errors(seg_features, lambda f, c, y, w, h: f.satellite_patch(c, y, w, h), config),
}
pd.DataFrame({name: d.groupby("reachable")["error_m"].agg(["median", "max", "count"]).stack() for name, d in identity.items()})

# Figures

In [ ]:
PALETTE = np.array([[40, 40, 40], [230, 30, 30], [255, 215, 0], [20, 160, 40], [200, 140, 255], [30, 110, 255]], np.uint8)

In [ ]:
fig_pose, axes = plt.subplots(POSE_FIGURE_FRAMES, 4, figsize=(16, 2.6 * POSE_FIGURE_FRAMES))
for row, frame in zip(axes, [frames[i] for i in np.linspace(0, len(frames) - 1, POSE_FIGURE_FRAMES).astype(int)]):
    width_m, height_m = PositionCorrelator(seg_features, config).footprint_m(frame)
    sat_rgb = satellite.crop_rotated(frame.position, frame.yaw_deg, width_m, height_m)
    drone_classes = seg_features.drone_probabilities(frame.image(), width_m, height_m).argmax(0)
    sat_classes = seg_features.probabilities.crop_rotated(frame.position, frame.yaw_deg, width_m, height_m, MATCH_GSD_M).argmax(-1)
    for ax, panel, title in zip(row, [np.asarray(frame.image()), sat_rgb, PALETTE[drone_classes], PALETTE[sat_classes]],
                                ["drone frame", "satellite at logged pose", "drone classes", "satellite classes"]):
        ax.imshow(panel); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=9)
    row[0].set_ylabel(f"{frame.name[-7:]}\nyaw {frame.yaw_deg:.0f}  alt {frame.altitude_m:.0f}", fontsize=8)
fig_pose.tight_layout()
fig_pose.savefig(FIGURES / "pose-drone-vs-satellite.png", dpi=90)

In [ ]:
fig_map, ax = plt.subplots(1, 2, figsize=(16, 6))
ax[0].imshow(satellite.resampled(MATCH_GSD_M).pixels)
ax[1].imshow(PALETTE[seg_features.probabilities.pixels.argmax(-1)])
for a in ax:
    xs, ys = zip(*(satellite.resampled(MATCH_GSD_M).to_px(f.position) for f in frames))
    a.plot(xs, ys, "c.", ms=4); a.set_xticks([]); a.set_yticks([])
ax[0].set_title("orthophoto at 0.385 m/px, frame positions"); ax[1].set_title("satellite class map (satellite BN)")
fig_map.tight_layout()
fig_map.savefig(FIGURES / "satellite-class-map.png", dpi=90)

In [ ]:
fig_reg, ax = plt.subplots(figsize=(5, 3.5))
ax.scatter(regression["peak_z"], regression["position_diff_m"])
ax.set_xlabel("peak z of the port's heatmap"); ax.set_ylabel("port vs 2025 position, m")
ax.set_title("port vs 2025, one point per draw"); ax.grid(alpha=.3)
fig_reg.tight_layout()
fig_reg.savefig(FIGURES / "regression-vs-2025.png", dpi=110)